# Commission Formation Dataset - QA & Exploration

Quality assurance notebook for reviewing the 2024-2029 Commission formation dataset outputs.

In [ ]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("../data/output")

# Load all tables
commissioners = pd.read_csv(OUTPUT_DIR / "commissioners.csv")
commitments = pd.read_csv(OUTPUT_DIR / "mission_letter_commitments.csv")
hearings = pd.read_csv(OUTPUT_DIR / "hearings.csv")
votes = pd.read_csv(OUTPUT_DIR / "investiture_vote.csv")
work_programme = pd.read_csv(OUTPUT_DIR / "work_programme_items.csv")
timeline = pd.read_csv(OUTPUT_DIR / "formation_timeline.csv")

print("Table sizes:")
for name, df in [("commissioners", commissioners), ("commitments", commitments),
                  ("hearings", hearings), ("votes", votes),
                  ("work_programme", work_programme), ("timeline", timeline)]:
    print(f"  {name}: {len(df)} rows x {len(df.columns)} cols")

## 1. Commissioners

In [ ]:
print(f"Countries: {commissioners['country'].nunique()}/27")
print(f"\nRole distribution:")
print(commissioners['role'].value_counts().to_string())
print(f"\nGender split:")
print(commissioners['gender'].value_counts().to_string())
print(f"\nParty groups:")
print(commissioners['ep_party_group'].value_counts().to_string())
commissioners[['full_name', 'country', 'portfolio_title', 'role']].head(10)

## 2. Mission Letter Commitments

In [ ]:
if not commitments.empty:
    print(f"Total commitments: {len(commitments)}")
    print(f"Commissioners covered: {commitments['commissioner_id'].nunique()}")
    print(f"\nBy extraction method:")
    print(commitments['extraction_method'].value_counts().to_string())
    print(f"\nBy confidence:")
    print(commitments['confidence'].value_counts().to_string())
    print(f"\nBy type:")
    print(commitments['commitment_type'].value_counts().to_string())
    print(f"\nCommitments per commissioner:")
    print(commitments.groupby('commissioner_name').size().sort_values(ascending=False).to_string())
else:
    print("No commitments extracted (mission letters may not have been downloaded).")

## 3. Investiture Vote

In [ ]:
print("Vote totals:")
print(votes['vote'].value_counts().to_string())
print(f"\nTotal MEPs: {len(votes)}")
print(f"Expected: 370 + 282 + 36 = 688")
print(f"\nVotes by party group:")
vote_by_group = votes.groupby(['ep_party_group', 'vote']).size().unstack(fill_value=0)
print(vote_by_group.to_string())
print(f"\nCountries represented: {votes['country'].nunique()}")

## 4. Hearings

In [ ]:
print(f"Total hearings: {len(hearings)}")
print(f"\nOutcomes:")
print(hearings['outcome'].value_counts().to_string())
print(f"\nHearings by date:")
print(hearings.groupby('hearing_date').size().to_string())
hearings[['commissioner_name', 'hearing_date', 'committees_responsible', 'outcome']]

## 5. Work Programme

In [ ]:
if not work_programme.empty:
    print(f"Total items: {len(work_programme)}")
    print(f"\nBy annex:")
    print(work_programme['annex'].value_counts().to_string())
    print(f"\nSample items:")
    print(work_programme[['annex', 'item_number', 'title']].head(15).to_string())
else:
    print("No work programme items extracted (PDF may not have been downloaded).")

## 6. Timeline

In [ ]:
print(f"Timeline: {len(timeline)} events")
timeline[['date', 'event_name', 'institution']]

## 7. Cross-Table Checks

In [ ]:
# Check FK integrity
comm_ids = set(commissioners['commissioner_id'])
hearing_ids = set(hearings['commissioner_id'])
print(f"Hearing commissioner IDs in commissioners table: {hearing_ids.issubset(comm_ids)}")

if not commitments.empty:
    commit_ids = set(commitments['commissioner_id'])
    print(f"Commitment commissioner IDs in commissioners table: {commit_ids.issubset(comm_ids)}")

# Check all 27 countries present
print(f"\nCountries with commissioner: {commissioners['country'].nunique()}/27")
print(f"Unique countries in votes: {votes['country'].nunique()}")